# 🧪 Laboratório Prático: Tokenização, Embeddings e Inferência Local em CPU
**Disciplina:** COM170 — Inteligência Artificial na Prática Acadêmica e Profissional  
**Quinzena 02:** Prompts, Atenção, Alucinação e Inferência  
**Perfil:** Estudo Prático Guiado (Passo a Passo)

---

## 🎯 Objetivos de Aprendizagem
Neste laboratório, você verá na prática o que acontece "por baixo do capô" de um modelo de linguagem (LLM):

1. **Tokenização:** Como o texto humano é fragmentado em unidades menores (tokens) e mapeado para números inteiros (*Token IDs*).
2. **Embeddings:** Como esses IDs são convertidos em vetores numéricos contínuos de alta dimensionalidade ($d_{model} = 768$ no GPT-2) que capturam relações semânticas.
3. **Inferência Local e Autoregressiva:** Como um modelo ultracompacto moderno (**Qwen 2.5 0.5B Instruct**) é carregado e gera texto token a token em tempo real na CPU do seu próprio computador.

---

## ⚙️ Pré-requisitos: Isolamento com Ambiente Virtual via `uv`
Para manter seu Python global limpo, o projeto utiliza o `uv` na raiz do repositório para gerenciar o ambiente virtual e as dependências:

### 1. Instalar as Dependências com `uv`
Abra o terminal na raiz do repositório (`EngComp-UNIVESP`) e execute:
```bash
uv add torch transformers ipykernel
```

### 2. Selecionar o Kernel no VS Code / Jupyter
> 💡 **Dica de execução:** No canto superior direito deste notebook no VS Code, clique em **Select Kernel** (ou *Selecionar Kernel*) $\rightarrow$ **Python Environments...** $\rightarrow$ selecione o interpretador `.venv` criado pelo `uv` na raiz.


In [1]:
print("Teste funcional")

Teste funcional


## 📦 Etapa 1: Importação das Bibliotecas e Verificação do Ambiente

Vamos importar as classes essenciais da biblioteca `transformers` do Hugging Face e a biblioteca `torch` (PyTorch).

In [3]:
import time
import torch
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    GPT2Tokenizer,
    GPT2Model
)

print(f"Versão do PyTorch: {torch.__version__}")
print(f"Dispositivo disponível para execução: {'CUDA (GPU)' if torch.cuda.is_available() else 'CPU'}")

Versão do PyTorch: 2.13.0+cpu
Dispositivo disponível para execução: CPU


---
## 🧠 Etapa 2: Investigação com Modelo Didático (GPT-2 Small - ~500 MB)

O **GPT-2 Small** (OpenAI) possui aproximadamente 124 milhões de parâmetros e uma dimensão oculta de embedding de 768. Ele é perfeito para estudo e inspeção de tensores porque é leve, rápido para carregar e seus componentes internos são amplamente documentados.

### 2.1 Carregamento do Tokenizador e do Modelo Base
Aqui carregamos o `GPT2Tokenizer` (que usa o algoritmo BPE — *Byte Pair Encoding*) e o `GPT2Model` (que contém os pesos do modelo e a tabela de embeddings).

In [6]:
# Carregamento do tokenizador e modelo didático GPT-2
llm_didatico_GPT2Model = "gpt2"

print("Baixando/Carregando tokenizador e modelo GPT-2...")
tokenizador_GPT2Tokenizer = GPT2Tokenizer.from_pretrained(llm_didatico_GPT2Model)
modelo_didatico = GPT2Model.from_pretrained(llm_didatico_GPT2Model)
print("Carregamento concluído com sucesso!")

Baixando/Carregando tokenizador e modelo GPT-2...


Loading weights: 100%|██████████| 148/148 [00:00<00:00, 3702.41it/s]

Carregamento concluído com sucesso!


### 2.2 Tokenização e Inspeção dos Token IDs

Vamos testar o prompt em inglês `"Senior Software Engineering"`.
Observe como o tokenizador decompõe o texto em tokens e os converte em números inteiros (*Token IDs*).

In [ ]:
prompt_ingles = "Senior Software Engineering"

# Codificação do prompt em tensores PyTorch
tokens_identificadores = tokenizador_GPT2Tokenizer.encode(prompt_ingles, return_tensors="pt")

print(f"Texto original: '{prompt_ingles}'")
print(f"Lista de Token IDs: {tokens_identificadores.tolist()[0]}")
print(f"Quantidade total de tokens gerados: {tokens_identificadores.shape[1]}")

# Inspeção token a token
print("\nDecomposição detalhada dos tokens:")
for indice_token, token_id in enumerate(tokens_identificadores[0]):
    token_texto = tokenizador_GPT2Tokenizer.decode(token_id)
    print(f"  • Token {indice_token + 1}: ID {token_id.item():<6} -> Representação textual: '{token_texto}'")


Texto original: 'Senior Software Engineering'
Lista de Token IDs: [31224, 10442, 14044]
Quantidade total de tokens gerados: 3

Decomposição detalhada dos tokens:
  • Token 1: ID 31224  -> Representação textual: 'Senior'
  • Token 2: ID 10442  -> Representação textual: ' Software'
  • Token 3: ID 14044  -> Representação textual: ' Engineering'


#### 💡 2.2.1 Adendo de Engenharia: Variações de Espaço, Caixa Alta/Baixa e Redundância de Vocabulário

Uma observação crítica de engenharia de software é: **por que o espaço faz parte do token e por que pequenas variações da mesma palavra viram tokens completamente diferentes?**

* **A Causa:** O algoritmo BPE (*Byte Pair Encoding*) é puramente estatístico e não normaliza o texto (`.lower()` ou `.strip()`). Ele preserva pontuações, maiúsculas e espaços para permitir que o modelo gere código de programação (onde identação e maiúsculas importam) e texto humano formatado com fidelidade exata (*lossless*).
* **O Custo (Trade-off):** Como consequência, palavras idênticas em contextos sintáticos diferentes (com espaço inicial, sem espaço, maiúscula, minúscula) recebem **IDs distintos** e ocupam **vetores de embedding separados** no vocabulário, duplicando parâmetros na rede neural.

Vamos rodar o teste comparativo abaixo para inspecionar essas variações na tabela do GPT-2:

In [9]:
# Teste de variações sintáticas da mesma palavra base ('Software')
variacoes_palavra = [
    "Software",    # Sem espaço inicial (início de frase)
    " Software",   # Com espaço inicial (meio de frase)
    " software",   # Minúsculo com espaço
    "SOFTWARE",    # Caixa alta sem espaço
    " SOFTWARE"    # Caixa alta com espaço
]

print("=======================================================================")
print("🔍 COMPARAÇÃO DE ENTRADAS NO VOCABULÁRIO DO GPT-2")
print("=======================================================================")
for palavra in variacoes_palavra:
    ids_gerados = tokenizador_GPT2Tokenizer.encode(palavra)
    print(f"Texto: {repr(palavra):<15} -> Token IDs: {str(ids_gerados):<10} | Qtd Tokens: {len(ids_gerados)}")
print("=======================================================================")
print("📌 Conclusão: Cada variação possui um Token ID único e um vetor de 768 dimensões próprio!")


🔍 COMPARAÇÃO DE ENTRADAS NO VOCABULÁRIO DO GPT-2
Texto: 'Software'      -> Token IDs: [25423]    | Qtd Tokens: 1
Texto: ' Software'     -> Token IDs: [10442]    | Qtd Tokens: 1
Texto: ' software'     -> Token IDs: [3788]     | Qtd Tokens: 1
Texto: 'SOFTWARE'      -> Token IDs: [15821, 37485] | Qtd Tokens: 2
Texto: ' SOFTWARE'     -> Token IDs: [47466]    | Qtd Tokens: 1
📌 Conclusão: Cada variação possui um Token ID único e um vetor de 768 dimensões próprio!


### 🔬 2.3 Comparação Didática: Inglês vs. Português

Como visto no material teórico da Quinzena 02, o GPT-2 foi treinado predominantemente com textos em inglês. Por isso, seu vocabulário é muito mais eficiente para o inglês do que para o português.

Vamos comparar a quantidade de tokens gerados para a tradução correspondente em português:

In [8]:
prompt_portugues = "Engenheiro de Software Sênior"

tokens_portugues = tokenizador_GPT2Tokenizer.encode(prompt_portugues, return_tensors="pt")

print(f"Texto em Inglês:    '{prompt_ingles}'   -> {tokens_identificadores.shape[1]} tokens")
print(f"Texto em Português: '{prompt_portugues}' -> {tokens_portugues.shape[1]} tokens")

print("\nDecomposição dos tokens em português:")
for indice_token, token_id in enumerate(tokens_portugues[0]):
    token_texto = tokenizador_GPT2Tokenizer.decode(token_id)
    print(f"  • Token {indice_token + 1}: ID {token_id.item():<6} -> '{token_texto}'")

Texto em Inglês:    'Senior Software Engineering'   -> 3 tokens
Texto em Português: 'Engenheiro de Software Sênior' -> 10 tokens

Decomposição dos tokens em português:
  • Token 1: ID 7936   -> 'Eng'
  • Token 2: ID 268    -> 'en'
  • Token 3: ID 258    -> 'he'
  • Token 4: ID 7058   -> 'iro'
  • Token 5: ID 390    -> ' de'
  • Token 6: ID 10442  -> ' Software'
  • Token 7: ID 311    -> ' S'
  • Token 8: ID 25792  -> 'ê'
  • Token 9: ID 77     -> 'n'
  • Token 10: ID 1504   -> 'ior'


### 2.4 Extração e Inspeção da Matriz de Embeddings (`wte`)

No GPT-2, a camada `wte` (*Word Token Embeddings*) funciona como uma tabela de consulta que mapeia cada *Token ID* inteiro para um vetor denso no espaço multidimensional $\mathbb{R}^{768}$.

Vamos inspecionar o formato do tensor resultante e visualizar os primeiros valores numéricos que compõem o significado de cada token:

In [10]:
# Passagem dos token IDs pela camada de Word Token Embeddings (wte)
matriz_embeddings = modelo_didatico.wte(tokens_identificadores)

print(f"Formato da matriz de embeddings: {list(matriz_embeddings.shape)}")
print("Estrutura do Tensor: [Batch Size = 1, Quantidade de Tokens = 3, Dimensão do Embedding = 768]")

# Exibição de uma amostra dos primeiros 5 valores numéricos do embedding de cada token
print("\nAmostra dos primeiros 5 valores numéricos de cada vetor:")
for indice_token, token_id in enumerate(tokens_identificadores[0]):
    token_texto = tokenizador_GPT2Tokenizer.decode(token_id)
    vetor_amostra = matriz_embeddings[0, indice_token, :5].detach().tolist()
    valores_formatados = [round(valor, 4) for valor in vetor_amostra]
    print(f"  • Token '{token_texto}' (ID {token_id.item()}): {valores_formatados} ... (total: 768 dimensões)")

Formato da matriz de embeddings: [1, 3, 768]
Estrutura do Tensor: [Batch Size = 1, Quantidade de Tokens = 3, Dimensão do Embedding = 768]

Amostra dos primeiros 5 valores numéricos de cada vetor:
  • Token 'Senior' (ID 31224): [0.0269, -0.0605, 0.3064, -0.159, -0.0473] ... (total: 768 dimensões)
  • Token ' Software' (ID 10442): [0.2042, -0.0464, 0.2047, -0.0994, 0.0842] ... (total: 768 dimensões)
  • Token ' Engineering' (ID 14044): [0.0691, -0.084, 0.1756, -0.0722, 0.0772] ... (total: 768 dimensões)


#### 🔍 2.4.1 Experimento Prático: Busca Direta na Matriz `wte.weight` por Token ID

A camada de embeddings do GPT-2 possui uma matriz de pesos contínua chamada `modelo_didatico.wte.weight`, com formato exato de **`[50257, 768]`**:
* **50.257 linhas**: Uma linha para cada Token ID do vocabulário (de 0 a 50.256).
* **768 colunas**: O vetor contínuo de significado de cada token.

Vamos inspecionar o ID específico **`25741`** e também sortear um **Token ID aleatório** para descobrir o texto correspondente e seus primeiros 5 valores decimais:

In [35]:
import random

# 1. Teste com o Token ID específico solicitado (25741)
id_especifico = 25741
texto_especifico = tokenizador_GPT2Tokenizer.decode([id_especifico])
vetor_especifico = modelo_didatico.wte.weight[id_especifico, :5].detach().tolist()
vetor_especifico_formatado = [round(valor, 4) for valor in vetor_especifico]

# 2. Teste com um Token ID sorteado aleatoriamente no vocabulário (0 a 50256)
id_aleatorio = random.randint(0, tokenizador_GPT2Tokenizer.vocab_size - 1)
texto_aleatorio = tokenizador_GPT2Tokenizer.decode([id_aleatorio])
vetor_aleatorio = modelo_didatico.wte.weight[id_aleatorio, :5].detach().tolist()
vetor_aleatorio_formatado = [round(valor, 4) for valor in vetor_aleatorio]

print("===================================================================================")
print("🔬 INSPEÇÃO DIRETA DA TABELA DE PESOS WTE (50257 x 768)")
print("===================================================================================")
print(f"🎯 Token ID Específico: {id_especifico}")
print(f"   • Representação Textual: {repr(texto_especifico)}")
print(f"   • 5 primeiros números da linha {id_especifico} na matriz wte: {vetor_especifico_formatado}")
print("-----------------------------------------------------------------------------------")
print(f"🎲 Token ID Aleatório Sorteado: {id_aleatorio}")
print(f"   • Representação Textual: {repr(texto_aleatorio)}")
print(f"   • 5 primeiros números da linha {id_aleatorio} na matriz wte: {vetor_aleatorio_formatado}")
print("===================================================================================")


🔬 INSPEÇÃO DIRETA DA TABELA DE PESOS WTE (50257 x 768)
🎯 Token ID Específico: 25741
   • Representação Textual: ' contradiction'
   • 5 primeiros números da linha 25741 na matriz wte: [0.0567, -0.0386, -0.0772, -0.1201, -0.0992]
-----------------------------------------------------------------------------------
🎲 Token ID Aleatório Sorteado: 34842
   • Representação Textual: ' isEnabled'
   • 5 primeiros números da linha 34842 na matriz wte: [0.1695, -0.2447, 0.121, -0.051, -0.2076]


---
## ⚡ Etapa 3: Inferência Local com Modelo Ultracompacto (Qwen 2.5 0.5B Instruct)

Agora vamos executar um modelo de linguagem moderno com capacidade de seguir instruções (*instruct*), contendo **500 milhões de parâmetros** (**Qwen2.5-0.5B-Instruct**), rodando diretamente na **CPU**.

### 3.1 Carregamento do Modelo e Tokenizador
O modelo ocupa aproximadamente ~350 MB a ~500 MB de memória RAM, sendo ideal para testes rápidos locais sem exigir GPU.

In [36]:
nome_modelo_ultra = "Qwen/Qwen2.5-0.5B-Instruct"

print(f"Baixando/Carregando {nome_modelo_ultra}...")
tokenizador_ultra = AutoTokenizer.from_pretrained(nome_modelo_ultra)
modelo_ultra = AutoModelForCausalLM.from_pretrained(nome_modelo_ultra)
print("Modelo carregado com sucesso na memória!")

Baixando/Carregando Qwen/Qwen2.5-0.5B-Instruct...


c:\Users\LKSFERREIRA\Documents\GitHub\EngComp-UNIVESP\.venv\Lib\site-packages\huggingface_hub\file_download.py:141: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\LKSFERREIRA\.cache\huggingface\hub\models--Qwen--Qwen2.5-0.5B-Instruct. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 290/290 [00:00<00:00, 3256.00it/s]


Modelo carregado com sucesso na memória!


### 3.2 Preparação do Prompt e Tokenização de Entrada

Vamos formular uma pergunta direta para testar a capacidade de síntese conceitual do modelo.

In [37]:
pergunta_estudo = "Explique em uma frase curta o que e um token em IA:"

entradas_tokenizadas = tokenizador_ultra(pergunta_estudo, return_tensors="pt")

print(f"Pergunta enviada: '{pergunta_estudo}'")
print(f"Quantidade de tokens de entrada (Prompt): {entradas_tokenizadas['input_ids'].shape[1]}")

Pergunta enviada: 'Explique em uma frase curta o que e um token em IA:'
Quantidade de tokens de entrada (Prompt): 15


### 3.3 Execução da Inferência e Medição de Desempenho em CPU

Vamos executar a geração autoregressiva com o método `generate()`, limitando a saída em 30 novos tokens (`max_new_tokens=30`).
Mediremos o tempo exato de resposta e calcularemos a taxa de tokens gerados por segundo (tokens/s).

In [44]:
# Marcação do tempo inicial
tempo_inicial = time.time()

# Geração autoregressiva dos tokens na CPU
with torch.no_grad():
    saidas_geradas = modelo_ultra.generate(
        **entradas_tokenizadas,
        max_new_tokens=500,
        pad_token_id=tokenizador_ultra.eos_token_id
    )

# Cálculo do tempo total decorrido
tempo_total_decorrido = time.time() - tempo_inicial

# Decodificação dos tokens gerados para texto legível
resposta_completa = tokenizador_ultra.decode(saidas_geradas[0], skip_special_tokens=True)

# Cálculo de métricas de desempenho
quantidade_tokens_gerados = saidas_geradas.shape[1] - entradas_tokenizadas['input_ids'].shape[1]
velocidade_tokens_por_segundo = quantidade_tokens_gerados / tempo_total_decorrido if tempo_total_decorrido > 0 else 0

print("==================================================")
print("📊 RESULTADOS DA EXECUÇÃO LOCAL (CPU)")
print("==================================================")
print(f"⏱️ Tempo de execução em CPU: {tempo_total_decorrido:.2f} segundos")
print(f"🚀 Novos tokens gerados: {quantidade_tokens_gerados} tokens")
print(f"⚡ Velocidade média: {velocidade_tokens_por_segundo:.2f} tokens/segundo")
print("--------------------------------------------------")
print("📝 Resposta Completa Gerada:")
print(resposta_completa)
print("==================================================")

📊 RESULTADOS DA EXECUÇÃO LOCAL (CPU)
⏱️ Tempo de execução em CPU: 70.81 segundos
🚀 Novos tokens gerados: 500 tokens
⚡ Velocidade média: 7.06 tokens/segundo
--------------------------------------------------
📝 Resposta Completa Gerada:
Explique em uma frase curta o que e um token em IA: Um token é uma subconjunto de um conjunto de dados, geralmente usado para representar um item específico ou característica específica. Em IA, os tokens são usados para processamento natural dos dados, transformando-os em informações mais úteis e pertinentes. Isso permite que as informações sejam interpretadas e utilizadas com maior eficiência.

Em resumo, um token é um elemento do vocabulário de entrada, utilizado pelo sistema para gerar novos resultados ou melhorar a interpretação de dados existentes. Portanto, um token é essencial para a compreensão e uso de informações no ambiente de IA. 

É importante notar que, embora o termo "token" seja frequentemente usado como um termo baseado em linguagem natur

---
## 📝 Etapa 4: Síntese Conceitual e Reflexão Crítica de Engenharia

Após rodar e investigar as entranhas dos modelos neste laboratório, consolidamos quatro grandes pilares de aprendizado fundamentados na prática e nos questionamentos de arquitetura:

### 1. 🔤 A Mecânica da Tokenização BPE e a Ausência de Normalização
* **Variações Sintáticas como Tokens Distintos:** Vimos na prática que `"Software"` (ID 25423), `" Software"` (ID 10442), `" software"` (ID 3788) e `" SOFTWARE"` (ID 47466) ocupam registros e vetores de 768 dimensões separados no vocabulário.
* **O Trade-off da Normalização:** Modelos clássicos de busca usavam `.lower()` e `.strip()`, mas modelos generativos modernos evitam a normalização destrutiva. O objetivo é manter uma representação estritamente *lossless* (sem perdas) para ser capaz de gerar código de programação funcional (onde identação de 4 espaços, maiúsculas e *camelCase* definem a sintaxe) e formatação de texto idêntica à humana.
* **Frequência Estatística vs. Gramática:** Comprovamos que `" SOFTWARE"` (com espaço) virou 1 token único porque expressões de licença de código aberto aparecem milhões de vezes na internet, enquanto `"SOFTWARE"` (sem espaço) precisou ser fatiado em 2 tokens (`'SOFT'` e `'WARE'`). O vocabulário reflete frequência estatística bruta da internet, e não bom senso gramatical.
* **Fronteira da Pesquisa (Embeddings Composicionais):** Discutimos como o desacoplamento de raiz semântica e modificadores sintáticos (*Factorized / Compositional Embeddings*) e modelos *Token-free* (como MEGABYTE e MambaByte) buscam eliminar esse desperdício de parâmetros em modelos de trilhões de pesos.

---

### 2. 🧠 A Estrutura dos Pesos: Tabela `wte`, Dimensões e Parâmetros
* **Diferenciação Conceitual:**
  * **Dimensão do Vetor ($d_{model} = 768$ no GPT-2):** A quantidade de coordenadas espaciais que compõem cada linha da matriz, desenhada para ser divisível perfeitamente pelas 12 cabeças de atenção ($12 \times 64 = 768$), alinhando-se à largura de banda dos registradores de GPU (potências de 2).
  * **Parâmetros/Pesos (124M no GPT-2, 500M no Qwen):** O total de números decimais que a rede guarda na memória (só a tabela `wte` do GPT-2 consome $50.257 \times 768 \approx 38,6$ milhões de pesos).
  * **Corpus de Treinamento:** Os bilhões ou trilhões de tokens de texto bruto lidos durante o treinamento para calibrar esses pesos.
* **Busca Direta por Linha:** Inspecionamos tensores reais acessando `modelo_didatico.wte.weight[id]`, descobrindo que o ID `25741` corresponde a `' contradiction'` e o `34842` a `' isEnabled'`.
* **Incompatibilidade entre Modelos:** Cada família de modelos possui seu próprio vocabulário e mapa numérico. Trocar o tokenizador sem retreinar o modelo gera saídas desconexas ou erros de índice fora da matriz.

---

### 3. ⏱️ Mecanismos de Parada (`EOS`), Janela de Contexto e Degeneração
* **O Token Especial `EOS` (*End of Sequence*):** O modelo só encerra a resposta de forma autônoma quando a rede neural atribui a maior probabilidade estatística ao token invisível de fim (`<|endoftext|>` ou `<|im_end|>`).
* **Degeneração em Texto Contínuo (*Raw Completion*):** Quando enviamos um prompt em texto puro com limite alto (`max_new_tokens = 500`), o modelo não recebe o gatilho de parada de um assistente de diálogo e entra em espirais retóricas de repetição (*"Em resumo..."*, *"Por fim..."*, *"No entanto..."*), gerando texto continuamente até bater no teto forçado pelo código.
* **Chat Templates:** Para que o modelo entenda a alternância de turnos de diálogo e encerre no ponto final em poucos tokens, a entrada deve ser estruturada com tags de chat (`<|im_start|>user...<|im_end|><|im_start|>assistant...`).

---

### 4. 💻 Viabilidade Prática de Inferência Local na CPU
* Comprovamos a viabilidade real de rodar modelos modernos ultracompactos (**Qwen 2.5 0.5B Instruct**) diretamente na **CPU**, atingindo uma taxa estável de **~7 tokens por segundo**, permitindo testes de arquitetura e validações de engenharia em ambiente local e sem custo de infraestrutura de nuvem.
